In [30]:
import yfinance as yf
import pandas as pd
df = yf.download("AAPL", start="2025-01-01", end="2025-07-17")  
# keep only the closing series
prices = df["Close"].reset_index(drop=True)
import torch
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",   # or one of the bolt models for speed
    device_map="cuda"             # or "cuda" if you have a GPU
)

[*********************100%***********************]  1 of 1 completed


In [27]:
# convert your series to a tensor
context = torch.tensor(prices.values[:-1], dtype=torch.float32)

# get median forecast for the next day
quantiles, mean = pipeline.predict_quantiles(
    context=context,
    prediction_length=1,
    quantile_levels=[0.5]      # just the median
)
pred_next = mean[0, 0].item()

In [28]:
pred_next

241.3013916015625

In [29]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# Prepare your data in long format:
# columns: 'item_id' (e.g. "AAPL"), 'timestamp', 'value' (closing price)
ts_df = TimeSeriesDataFrame.from_data_frame(
    df.reset_index().rename(columns={"Date": "timestamp", "Close": "value"}),
    id_column="ticker",
    timestamp_column="timestamp",
    target="value",
)

# Fine-tune Chronos-Bolt (small) on your AAPL series:
predictor = TimeSeriesPredictor(
    prediction_length=1,
    eval_metric="MAE"
).fit(
    train_data=ts_df,
    presets="bolt_small",     # zero-shot + optional fine-tuning
)

# Now get next-day point forecast:
pred = predictor.predict(ts_df)
print(pred.iloc[0, 0])

ModuleNotFoundError: No module named 'autogluon'

In [49]:
df = yf.download("AAPL", start="2025-01-01", end="2025-07-17")["Close"]

[*********************100%***********************]  1 of 1 completed


In [52]:
df.values.flatten()

array([243.26319885, 242.77436829, 244.41041565, 241.62713623,
       242.11595154, 236.28004456, 233.83592224, 232.71861267,
       237.29756165, 227.71069336, 229.42655945, 222.10421753,
       223.29136658, 223.1217804 , 222.24388123, 229.30685425,
       237.68663025, 238.78398132, 237.01823425, 235.43208313,
       227.46128845, 232.23977661, 231.9105835 , 232.6587677 ,
       227.08221436, 227.35185242, 232.31535339, 236.55978394,
       241.21368408, 244.27966309, 244.14984131, 244.54930115,
       245.50805664, 245.22842407, 246.77639771, 246.71646118,
       240.04521179, 236.98922729, 241.52326965, 237.71826172,
       235.62101746, 235.43127441, 235.02180481, 238.75691223,
       227.18208313, 220.55078125, 216.6958313 , 209.40539551,
       213.2104187 , 213.71974182, 212.41145325, 214.95811462,
       213.8196106 , 217.98414612, 220.44091797, 223.45697021,
       221.23986816, 223.55683899, 217.61462402, 221.83909607,
       222.89770508, 223.5967865 , 202.92390442, 188.13

In [31]:
tickers = [
    # Technology
    "AAPL",  # Apple Inc.
    "MSFT",  # Microsoft Corporation
    "GOOGL", # Alphabet Inc. (Google)
    "AMZN",  # Amazon.com Inc.
    "TSLA",  # Tesla Inc.
    "NVDA",  # NVIDIA Corporation
    "META",  # Meta Platforms Inc. (Facebook)
    "AMD",   # Advanced Micro Devices Inc.
    "INTC",  # Intel Corporation
    "CSCO",  # Cisco Systems Inc.

    # Finance
    "BRK-B", # Berkshire Hathaway Inc.
    "JPM",   # JPMorgan Chase & Co.
    "V",     # Visa Inc.
    "MA",    # Mastercard Inc.
    "GS",    # Goldman Sachs Group Inc.

    # Consumer Goods & Retail
    "NFLX",  # Netflix Inc.
    "DIS",   # Walt Disney Co.
    "PG",    # Procter & Gamble Co.
    "PEP",   # PepsiCo Inc.
    "KO",    # The Coca-Cola Company
    "MCD",   # McDonald's Corporation
    "WMT",   # Walmart Inc.
    "TGT",   # Target Corporation
    "NKE",   # Nike Inc.

    # Energy & Industrials
    "XOM",   # Exxon Mobil Corporation
    "CVX",   # Chevron Corporation
    "BA",    # Boeing Co.
    "CAT",   # Caterpillar Inc.

    # Healthcare & Pharmaceuticals
    "UNH",   # UnitedHealth Group Incorporated
    "PFE",   # Pfizer Inc.
    "JNJ",   # Johnson & Johnson
    "LLY",   # Eli Lilly and Co.
    "MRNA",  # Moderna Inc.
]

In [45]:
all_dfs = []
for t in tickers:
    df = yf.download(t, start="2015-01-01", end="2025-01-17")[["Close"]].dropna()
    all_dfs.append(df.reset_index(drop=True))
long_df = pd.concat(all_dfs, ignore_index=True)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [46]:
long_df.values

array([[24.28857994,         nan,         nan, ...,         nan,
                nan,         nan],
       [23.6043396 ,         nan,         nan, ...,         nan,
                nan,         nan],
       [23.60655403,         nan,         nan, ...,         nan,
                nan,         nan],
       ...,
       [        nan,         nan,         nan, ...,         nan,
                nan, 34.45999908],
       [        nan,         nan,         nan, ...,         nan,
                nan, 34.77000046],
       [        nan,         nan,         nan, ...,         nan,
                nan, 33.75999832]])

In [62]:
from pathlib import Path
from typing import List, Union

import numpy as np
from gluonts.dataset.arrow import ArrowWriter


def convert_to_arrow(
    path: Union[str, Path],
    time_series: Union[List[np.ndarray], np.ndarray],
    compression: str = "lz4",
):
    """
    Store a given set of series into Arrow format at the specified path.

    Input data can be either a list of 1D numpy arrays, or a single 2D
    numpy array of shape (num_series, time_length).
    """
    assert isinstance(time_series, list) or isinstance(time_series, np.ndarray)

    # Set an arbitrary start time
    start = np.datetime64("2000-01-01 00:00", "s")

    dataset = [
        {"start": start, "target": ts} for ts in time_series
    ]

    ArrowWriter(compression=compression).write_to_file(
        dataset,
        path=path,
    )

convert_to_arrow("./noise-data.arrow",time_series=df.values.flatten())

In [57]:
type(df.values.flatten())

numpy.ndarray

In [58]:
isinstance(df.values.flatten(), np.ndarray)

True

In [63]:
# List of training data files
training_data_paths:
- "/path/to/kernelsynth-data.arrow"
# Mixing probability of each dataset file
probability:
- 1.0

SyntaxError: invalid syntax (651213112.py, line 2)

In [64]:
CUDA_VISIBLE_DEVICES=0 python training/train.py --config /path/to/modified/config.yaml \
    --model-id amazon/chronos-t5-small \
    --no-random-init \
    --max-steps 1000 \
    --learning-rate 0.001

SyntaxError: invalid syntax (3259199332.py, line 1)